In [ ]:
###############################################################################
# IMPORTS
###############################################################################

import os
import sys
import time
import glob
import math
import json
import pickle
import argparse
import warnings
import statistics
from collections import defaultdict
from typing import Dict, List, Literal, Optional, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import dimod
from dimod import BinaryQuadraticModel, make_quadratic, BINARY
from dwave.system import DWaveSampler, EmbeddingComposite, FixedEmbeddingComposite
import minorminer
import dwave_networkx as dnx
from neal import SimulatedAnnealingSampler

In [ ]:
###############################################################################
# SAMPLER HANDLES
###############################################################################

sampler_quantum = DWaveSampler(solver='Advantage2_system1.12', token='<INPUT D-WAVE TOKEN HERE>')
sampler_classic = SimulatedAnnealingSampler()

In [ ]:
###############################################################################
# BROOKS' THEOREM BOUND
###############################################################################

def brooks_bound(G: nx.Graph) -> int:
    
    """
    Upper bound on chromatic number via Brooks' theorem.

    For connected G that is neither complete nor an odd cycle: χ(G) ≤ Δ(G).
    For complete graphs K_n:  χ = n = Δ + 1.
    For odd cycles C_{2k+1}: χ = 3 = Δ + 1.
    For disconnected graphs:  max over components.
    """
    
    if G.number_of_nodes() == 0:
        return 0
    if G.number_of_nodes() == 1:
        return 1
    if G.number_of_edges() == 0:
        return 1

    max_chi = 1
    for comp in nx.connected_components(G):
        H = G.subgraph(comp).copy()
        n = H.number_of_nodes()
        delta = max(d for _, d in H.degree()) if n > 0 else 0

        if delta <= 1:
            chi = min(2, n)
        elif delta == 2:
            # Δ=2 → path or cycle.  Odd cycle needs 3 colors.
            if H.number_of_edges() == n and n % 2 == 1:
                chi = 3
            else:
                chi = 2
        else:
            # Complete graph check: |E| = n(n-1)/2
            if H.number_of_edges() == n * (n - 1) // 2:
                chi = n
            else:
                chi = delta
        max_chi = max(max_chi, chi)

    return max_chi

In [ ]:
###############################################################################
# ONE-HOT ENCODING
###############################################################################

def build_onehot_qubo(G: nx.Graph, C: int):
    
    """
    One-hot QUBO for Minimum Graph Coloring.

    Variables
    ---------
    x_{v,c} : 1 iff vertex v gets color c     (|V|·C  variables)
    y_c     : 1 iff color c is used            (C       variables)

    Hamiltonian
    -----------
    H = A1 Σ_v (1 - Σ_c x_{v,c})²            one-hot
      + A2 Σ_{(u,v)∈E} Σ_c x_{u,c} x_{v,c}   adjacency
      + A3 Σ_c y_c                           minimize colors
      + A4 Σ_v Σ_c x_{v,c}(1 - y_c)          link

    Penalties:  A3=1,  A4=|V|+1,  A2=C·A4+1,  A1=(|E|+1)·A2
    Logical qubits: C·(|V|+1)

    Returns (bqm, info_dict).
    """
    
    nodes = sorted(G.nodes())
    edges = list(G.edges())
    n, m  = len(nodes), len(edges)
    nid   = {v: i for i, v in enumerate(nodes)}

    A3 = 1.0
    A4 = float(n + 1)
    A2 = C * A4 + 1.0
    A1 = (m + 1) * A2

    xv = lambda v, c: f"x_{nid[v]}_{c}"
    yc = lambda c: f"y_{c}"

    bqm = BinaryQuadraticModel(vartype=BINARY)

    # H1: one-hot   A1·Σ_v [ 1 - Σ_c x_{v,c} + 2·Σ_{c<c'} x_{v,c} x_{v,c'} ]
    for v in nodes:
        for c in range(C):
            bqm.add_linear(xv(v, c), -A1)
        for c1 in range(C):
            for c2 in range(c1 + 1, C):
                bqm.add_quadratic(xv(v, c1), xv(v, c2), 2.0 * A1)

    # H2: adjacency
    for u, v in edges:
        for c in range(C):
            bqm.add_quadratic(xv(u, c), xv(v, c), A2)

    # H3: minimize colors
    for c in range(C):
        bqm.add_linear(yc(c), A3)

    # H4: link  A4·[ x_{v,c} - x_{v,c} y_c ]
    for v in nodes:
        for c in range(C):
            bqm.add_linear(xv(v, c), A4)
            bqm.add_quadratic(xv(v, c), yc(c), -A4)

    bqm.offset += A1 * n  # constant from (1 - ...)^2

    info = {
        "encoding": "one-hot",
        "num_colors": C,
        "num_logical_qubits": C * (n + 1),
        "num_logical_qubits_pre_quad": C * (n + 1),
        "num_auxiliary": 0,
        "penalties": {"A1": A1, "A2": A2, "A3": A3, "A4": A4},
    }
    return bqm, info

In [ ]:
###############################################################################
# BINARY (LOGARITHMIC) ENCODING
###############################################################################

def _poly_mul(p1: dict, p2: dict) -> dict:
    
    """
    Multiply two multilinear polynomials over binary variables.
    Represented as  {frozenset_of_vars: coeff}.
    Variables from different bit-positions never collide.
    """
    
    result = defaultdict(float)
    for v1, c1 in p1.items():
        for v2, c2 in p2.items():
            result[v1 | v2] += c1 * c2
    return dict(result)


def _expand_edge_product(ui: int, vi: int, L: int) -> dict:
    
    """
    Expand  Π_{k=0}^{L-1} (1 - b_{u,k} - b_{v,k} + 2·b_{u,k}·b_{v,k})
    = Π_k XNOR(b_{u,k}, b_{v,k}).

    Returns {frozenset((vertex_idx, bit_idx), ...): coeff}.
    """
    
    poly = {frozenset(): 1.0}
    for k in range(L):
        factor = {
            frozenset():                  1.0,
            frozenset({(ui, k)}):        -1.0,
            frozenset({(vi, k)}):        -1.0,
            frozenset({(ui, k), (vi, k)}): 2.0,
        }
        poly = _poly_mul(poly, factor)
    return {k: v for k, v in poly.items() if abs(v) > 1e-15}


def build_binary_hubo(G: nx.Graph, C: int):
    
    """
    Binary HUBO for Minimum Graph Coloring:

        H(b) = A · Σ_{(u,v)∈E} Π_k XNOR(b_{u,k}, b_{v,k})
             + Σ_k P_k · Σ_v b_{v,k}

    Penalties:
        P_k = (|V|+1)^{k-1}     for k = 1, …, L
        A   = |V| · Σ_k P_k + 1

    L = ⌈log₂ C⌉  bits per vertex.   Pre-quad logical qubits = |V|·L.

    Returns (hubo_dict, info_dict).
    hubo_dict: {tuple_of_var_names: coeff}  (dimod.make_quadratic format).
    """
    
    nodes = sorted(G.nodes())
    edges = list(G.edges())
    n = len(nodes)
    nid = {v: i for i, v in enumerate(nodes)}

    L = max(1, math.ceil(math.log2(C))) if C > 1 else 1

    # Penalties
    Pk = [(n + 1) ** k for k in range(L)]        # P_1=1, P_2=n+1, …
    A  = float(n * sum(Pk) + 1)

    bvar = lambda vi, k: f"b_{vi}_{k}"

    hubo = defaultdict(float)

    # Adjacency term
    for u, v in edges:
        poly = _expand_edge_product(nid[u], nid[v], L)
        for fset, coeff in poly.items():
            key = tuple(sorted(bvar(vi, ki) for vi, ki in fset))
            hubo[key] += A * coeff

    # Lexicographic penalty
    for k in range(L):
        for v in nodes:
            key = (bvar(nid[v], k),)
            hubo[key] += Pk[k]

    offset = hubo.pop((), 0.0)
    hubo = {k: v for k, v in hubo.items() if abs(v) > 1e-15}

    max_deg = max((len(k) for k in hubo), default=0)

    info = {
        "encoding": "binary",
        "num_colors": C,
        "L": L,
        "num_logical_qubits_pre_quad": n * L,
        "hubo_max_degree": max_deg,
        "hubo_num_terms": len(hubo),
        "offset": offset,
        "penalties": {"A": A, "Pk": Pk},
    }
    return dict(hubo), info


def quadratize_binary(hubo: dict, info: dict, G: nx.Graph):
    
    """
    Quadratize via dimod.make_quadratic (Rosenberg reduction).

    Penalty strength:  M = A + |V|·Σ_k P_k + 1.

    Returns (bqm, updated_info).
    """
    
    n  = G.number_of_nodes()
    A  = info["penalties"]["A"]
    Pk = info["penalties"]["Pk"]
    M  = A + n * sum(Pk) + 1.0

    bqm = make_quadratic(hubo, strength=M, vartype=BINARY)
    bqm.offset += info.get("offset", 0.0)

    n_pre  = info["num_logical_qubits_pre_quad"]
    n_post = bqm.num_variables
    n_aux  = n_post - n_pre

    L = info["L"]
    theoretical_aux = G.number_of_edges() * max(0, 2 * L - 2)

    info.update({
        "num_logical_qubits_post_quad": n_post,
        "num_auxiliary": n_aux,
        "theoretical_auxiliary": theoretical_aux,
        "quad_strength": M,
    })
    return bqm, info

In [ ]:
###############################################################################
# SOLVER INTERFACE
###############################################################################

def get_sampler(use_dwave=False, token=None, solver=None,
                use_inhomogeneous_driving=False):
    
    """
    Return (sampler, sampler_type) for downstream routing.

    If use_dwave is True and the Ocean SDK is importable, a DWaveSampler is
    returned wrapped in an EmbeddingComposite (sampler_type='dwave') or a
    raw DWaveSampler when inhomogeneous driving is requested
    (sampler_type='dwave_ihd').  Falls back to SimulatedAnnealingSampler
    (sampler_type='simulated') on any failure.
    """
    
    if use_dwave and DWAVE_AVAILABLE:
        try:
            kw = {}
            if token:
                kw["token"] = token
            if solver:
                kw["solver"] = solver
            raw = DWaveSampler(**kw)
            if use_inhomogeneous_driving:
                # Return raw sampler — sample_dwave() handles
                # embedding + offsets internally
                return raw, "dwave_ihd"
            else:
                return EmbeddingComposite(raw), "dwave"
        except Exception as e:
            print(f"  [WARN] D-Wave unavailable ({e}); falling back to SA.")
    return SimulatedAnnealingSampler(), "simulated"

In [ ]:
###############################################################################
# MINOR-EMBEDDING HELPER
###############################################################################

def embed(encoding: str, output_dir: str, name: str,
          sampler: "DWaveSampler", bqm: BinaryQuadraticModel, info: dict = None):
    
    """
    Find a minor embedding of bqm onto the sampler's hardware graph and
    persist it to disk as JSON.

    Parameters
    ----------
    encoding : str
        Tag identifying the encoding ('onehot' or 'logarithmic').  Included
        in the output filename.
    output_dir : str
        Directory in which to write the embedding JSON.
    name : str
        Graph instance name (basename of the .pkl, stripped of extension).
        Included in the output filename.
    sampler : DWaveSampler
        Raw D-Wave sampler whose target topology is used for embedding.
    bqm : BinaryQuadraticModel
        Source model whose interaction graph is the embedding source.
    info : dict, optional
        Currently unused; accepted for interface consistency.

    Writes  <output_dir>/<name>_<encoding>.json.  Raises RuntimeError if
    minorminer fails to find any embedding.
    """
    
    if not DWAVE_AVAILABLE:
        raise ImportError(
            "dwave-system required. Install via: pip install dwave-ocean-sdk"
        )

    source_edgelist = list(bqm.quadratic.keys())
    target_edgelist = sampler.structure.edgelist
    embedding = minorminer.find_embedding(source_edgelist, target_edgelist)
    if not embedding:
        raise RuntimeError(
            f"Failed to embed BQM ({bqm.num_variables} vars) onto QPU."
        )

    with open(f'{output_dir}/{name}_{encoding}.json', 'w') as f:
        json.dump(embedding, f, indent=4)

In [ ]:
###############################################################################
# CONFIGURATION AND GRAPH LOADING
###############################################################################

# ── Options ───────────────────────────────────────────────────────────────────
graph_dir                 = "<INPUT GRAPH DIRECTORY HERE>"
output_dir                = "<INPUT OUTPUT DIRECTORY HERE>"
use_dwave                 = True
dwave_token               = "<INPUT D-WAVE TOKEN HERE>"
dwave_solver              = "Advantage2_system1.12"
use_inhomogeneous_driving = True
max_nodes                 = 100

# ── Optional: real D-Wave hardware ────────────────────────────────────────────
try:
    from dwave.system import DWaveSampler, EmbeddingComposite, FixedEmbeddingComposite
    import minorminer
    import dwave_networkx as dnx
    DWAVE_AVAILABLE = True
except ImportError:
    DWAVE_AVAILABLE = False

# ── Load graphs ───────────────────────────────────────────────────────────────
os.makedirs(output_dir, exist_ok=True)

pkls = sorted(glob.glob(os.path.join(graph_dir, "*.pkl")))
if not pkls:
    sys.exit(f"No .pkl files in {graph_dir}")

graphs = []
for p in pkls:
    basename = os.path.splitext(os.path.basename(p))[0]
    with open(p, "rb") as f:
        obj = pickle.load(f)
    # Handle both single graphs and lists/tuples of graphs
    if isinstance(obj, nx.Graph):
        graphs.append((basename, obj))
    elif isinstance(obj, (list, tuple)):
        for i, item in enumerate(obj):
            if isinstance(item, nx.Graph):
                graphs.append((f"{basename}_{i}", item))
            else:
                print(f"  [WARN] {basename}[{i}]: unexpected type "
                      f"{type(item).__name__}, skipping")
    else:
        print(f"  [WARN] {basename}: unexpected type "
              f"{type(obj).__name__}, skipping")
print(f"Loaded {len(graphs)} graphs from {graph_dir}")

orig = len(graphs)
graphs = [(n, G) for n, G in graphs if G.number_of_nodes() <= max_nodes]
if len(graphs) < orig:
    print(f"Filtered to {len(graphs)} graphs (≤{max_nodes} nodes)")

# ── Initialize sampler ────────────────────────────────────────────────────────
tok = dwave_token or os.environ.get("DWAVE_API_TOKEN")
sampler, stype = get_sampler(
    use_dwave, tok,
    solver=dwave_solver,
    use_inhomogeneous_driving=use_inhomogeneous_driving,
)
print(f"Sampler: {stype}")

In [ ]:
###############################################################################
# COMPUTE AND SAVE PER-ENCODING MINOR EMBEDDINGS
###############################################################################

for name, G in graphs:
    try:
        C        = brooks_bound(G)
        bqm_oh   = build_onehot_qubo(G, C)
        hubo_bin = build_binary_hubo(G, C)
        bqm_bin  = quadratize_binary(hubo_bin[0], hubo_bin[1], G)

        embed("onehot",      output_dir, name, sampler, bqm_oh[0],  bqm_oh[1])
        embed("logarithmic", output_dir, name, sampler, bqm_bin[0], bqm_bin[1])

    except Exception as exc:
        print(f"  [ERR] {name}: {exc}")
        import traceback; traceback.print_exc()

In [ ]:
###############################################################################
# AGGREGATE CHAIN-LENGTH STATISTICS PER GRAPH AND ENCODING
###############################################################################

results = []

for name, G in graphs:
    try:
        nodes   = G.number_of_nodes()
        edges   = G.number_of_edges()
        avg_deg = sum(dict(G.degree()).values()) / G.number_of_nodes()
        max_deg = max(dict(G.degree()).values())

        # ── One-hot embedding stats ───────────────────────────────────────────
        with open(os.path.join(output_dir, f"{name}_onehot.json")) as f_onehot:
            data_onehot = json.load(f_onehot)

        onehot_lengths  = [len(v) for v in data_onehot.values()]
        onehot_total    = sum(onehot_lengths)
        onehot_average  = statistics.mean(onehot_lengths)
        onehot_variance = statistics.variance(onehot_lengths)

        # ── Logarithmic embedding stats ───────────────────────────────────────
        with open(os.path.join(output_dir, f"{name}_logarithmic.json")) as f_logarithmic:
            data_logarithmic = json.load(f_logarithmic)

        logarithmic_lengths  = [len(v) for v in data_logarithmic.values()]
        logarithmic_total    = sum(logarithmic_lengths)
        logarithmic_average  = statistics.mean(logarithmic_lengths)
        logarithmic_variance = statistics.variance(logarithmic_lengths)

        results.append((
            name, nodes, edges, avg_deg, max_deg,
            onehot_total,      onehot_average,      onehot_variance,
            logarithmic_total, logarithmic_average, logarithmic_variance,
        ))

    except Exception as exc:
        print(f"  [ERR] {name}: {exc}")
        import traceback; traceback.print_exc()

if not results:
    sys.exit("No results.")

df = pd.DataFrame(results, columns=[
    "name",
    "nodes",
    "edges",
    "avg_deg",
    "max_deg",
    "onehot_total",
    "onehot_average",
    "onehot_variance",
    "logarithmic_total",
    "logarithmic_average",
    "logarithmic_variance",
])

In [ ]:
###############################################################################
# WRITE RESULTS TO CSV
###############################################################################

csv_path = os.path.join(output_dir, "embedding_results.csv")
df.to_csv(csv_path, index=False)
print(f"Results → {csv_path}")